# Fresh vs Rotten Fruit Classification — Preprocessing

This notebook prepares the fruit image dataset for model training.

## Preprocessing goals

1. Build a labeled image index.
2. Create the required 70% training, 15% validation, and 15% testing split.
3. Preserve class proportions using stratified splitting.
4. Convert images to RGB.
5. Resize images to a common input size.
6. Normalize pixel values before model training.

In [ ]:
import kagglehub
from pathlib import Path

# Access the Kaggle dataset
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

DATASET_ROOT = Path(dataset_path) / "dataset"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

NUM_CLASSES = len(CLASS_NAMES)

print("Dataset root:", DATASET_ROOT)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

## 1. Build Image Index

In [ ]:
records = []

for split_name in ["train", "test"]:
    split_dir = DATASET_ROOT / split_name

    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name

        for image_path in class_dir.iterdir():
            if (
                image_path.is_file()
                and image_path.suffix.lower() in IMAGE_EXTENSIONS
            ):
                records.append(
                    {
                        "original_split": split_name,
                        "class_name": class_name,
                        "image_path": str(
                            image_path.relative_to(DATASET_ROOT)
                        ),
                    }
                )

print("Total image records:", len(records))

## 2. Create Image DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

print("DataFrame shape:", df.shape)
print("\nFirst 5 records:")
display(df.head())

In [ ]:
print("\nClass distribution:")
print(df["class_name"].value_counts())

print("\nOriginal split distribution:")
print(df["original_split"].value_counts())

## 3. Stratified Train / Validation / Test Split

The dataset is divided into 70% training, 15% validation, and 15% testing data.

Stratified splitting is used so that the class distribution is approximately preserved across all three subsets.

In [ ]:
from sklearn.model_selection import train_test_split

# First split:
# 70% training
# 30% temporary set
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class_name"],
    random_state=42,
)

# Second split:
# 15% validation
# 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class_name"],
    random_state=42,
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

print("Training images  :", len(train_df))
print("Validation images:", len(val_df))
print("Testing images   :", len(test_df))
print("Total images     :", len(train_df) + len(val_df) + len(test_df))

In [ ]:
print("Training distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting distribution:")
print(test_df["class_name"].value_counts())

## 4. Save Dataset Splits

In [ ]:
from pathlib import Path

DATA_OUTPUT_DIR = Path("/content/fruit_split_metadata")
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df[["image_path", "class_name"]].to_csv(
    DATA_OUTPUT_DIR / "train.csv",
    index=False
)

val_df[["image_path", "class_name"]].to_csv(
    DATA_OUTPUT_DIR / "validation.csv",
    index=False
)

test_df[["image_path", "class_name"]].to_csv(
    DATA_OUTPUT_DIR / "test.csv",
    index=False
)

print("Split files created:")
for file_path in sorted(DATA_OUTPUT_DIR.iterdir()):
    print(file_path.name, "→", file_path.stat().st_size, "bytes")

In [ ]:
print("Train CSV rows      :", len(train_df))
print("Validation CSV rows :", len(val_df))
print("Test CSV rows       :", len(test_df))